# PGA Tour player stats for one season

In [ ]:
%pip install requests pandas python-dotenv pymysql sqlalchemy

In [58]:
import os
import json #handle API response
import requests #send requests to API
import pandas as pd #data transformation
from mysql import connector #connect to mySQL
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [59]:
load_dotenv()

True

In [60]:
API_KEY = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
ACCESS_LEVEL = "trial"
GOLF_TOUR = "pga"
LANGUAGE_CODE = "en"
SEASON_YEAR = 2026
FORMAT = "json"

In [61]:
url = f"https://api.sportradar.com/golf/{ACCESS_LEVEL}/{GOLF_TOUR}/v3/{LANGUAGE_CODE}/{SEASON_YEAR}/players/statistics.{FORMAT}"
headers = {
    "x-api-key": API_KEY,
    "x-api-host": API_HOST
}
querystring = {
    "season": SEASON_YEAR,
    "tour": GOLF_TOUR
}

# ------- EXTRACT --------

In [62]:
response = requests.get(url=url,
                        headers=headers,
                        params=querystring)

In [63]:
payload = response.json()

In [ ]:
formatted_response = json.dumps(payload, indent=4)
print(formatted_response)

# --------- TRANSFORM ------------

In [ ]:
players_list = payload["players"]
players_list

In [ ]:
formatted_players_list = json.dumps(players_list, indent=4)
print(formatted_players_list)

In [67]:
McIlroy_wins = players_list[0]["statistics"]["first_place"]
print(McIlroy_wins)

1


In [68]:
rows = []
column_names = ['first_name', 'last_name', 'country', 'world_rank', 'events_played', 'first_place', 'second_place', 'top_10', 'top_25', 'cuts_made', 'fed_ex_rank', 'strokes_gained_total']

for player in players_list:
    first_name              = player['first_name']
    last_name               = player['last_name']
    country                 = player['country']
    world_rank              = player['statistics']['world_rank']
    points_rank             = player['statistics']['points_rank']
    events_played           = player['statistics']['events_played']
    first_place             = player['statistics']['first_place']
    second_place            = player['statistics']['second_place']
    top_10                  = player['statistics']['top_10']
    top_25                  = player['statistics']['top_25']
    cuts_made               = player['statistics']['cuts_made']
    FedEx_rank              = player['statistics']['points_rank']
    strokes_gained_total    = player['statistics']['strokes_gained_total']

    tuple_of_player_data = (first_name, last_name, country, world_rank, events_played, first_place, second_place, top_10, top_25, cuts_made, FedEx_rank, strokes_gained_total)

    rows.append(tuple_of_player_data)



In [69]:
print(rows[0])

('Rory', 'McIlroy', 'NORTHERN IRELAND', 2, 14, 1, 1, 6, 9, 14, 4, 1.341)


In [70]:
df_main = pd.DataFrame(rows, columns=column_names)

In [71]:
rows = []
column_names = ['first_name', 'last_name', 'id', 'drive_acc', 'gir_pct', 'sand_save_pct', 'scram_pct', 'scoring_avg', 'putts_per_round', 'strokes_gained']

for stats in players_list:
    first_name              = stats['first_name']
    last_name               = stats['last_name']
    id                      = stats['id']
    drive_acc               = stats['statistics']['drive_acc']
    gir_pct                 = stats['statistics']['gir_pct']
    sand_save_pct           = stats['statistics']['sand_saves_pct']
    scram_pct               = stats['statistics']['scrambling_pct']
    scoring_avg             = stats['statistics']['scoring_avg']
    putts_per_round         = stats['statistics']['putt_avg'] * 18
    strokes_gained          = stats['statistics']['strokes_gained_total']

    tuple_of_player_stats_data = (first_name, last_name, id, drive_acc, gir_pct, sand_save_pct, scram_pct, scoring_avg, putts_per_round, strokes_gained)

    rows.append(tuple_of_player_stats_data)

In [72]:
df_stats = pd.DataFrame(rows, columns=column_names)

In [73]:
df_main.head()

,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total
0,Rory,McIlroy,NORTHERN IRELAND,2,14,1,1,6,9,14,4,1.341
1,Jackson,Suber,UNITED STATES,94,20,0,0,3,6,13,74,0.475
2,Nicolai,Højgaard,DENMARK,42,21,0,2,4,8,16,35,0.691
3,Kris,Ventura,NORWAY,195,19,0,1,3,5,7,107,0.195
4,Gordon,Sargent,UNITED STATES,331,18,0,1,2,3,7,132,-0.694


In [74]:
df_stats.head()

,first_name,last_name,id,drive_acc,gir_pct,sand_save_pct,scram_pct,scoring_avg,putts_per_round,strokes_gained
0,Rory,McIlroy,da226913-b804-48de-adbf-96e956eb75ac,54.65,68.62,56.04,59.34,69.342,31.302,1.341
1,Jackson,Suber,da8b4888-9330-46d0-8f8a-f76fae949dba,60.78,70.68,54.55,61.52,70.310,31.968,0.475
2,Nicolai,Højgaard,7a4b2b40-4675-41d9-86fd-cc6a61a1d3b2,52.64,65.39,50.00,60.52,70.168,31.482,0.691
3,Kris,Ventura,a9053c60-3cc0-48cb-bcf2-4848963f352d,63.24,69.73,53.85,62.17,70.693,31.932,0.195
4,Gordon,Sargent,9b625995-cfb6-4ae6-91bb-b736b7d56bf7,60.15,66.31,51.61,55.79,71.602,31.662,-0.694


In [75]:
df_main.info()
df_stats.info()

<class 'pandas.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   first_name            219 non-null    str    
 1   last_name             219 non-null    str    
 2   country               219 non-null    str    
 3   world_rank            219 non-null    int64  
 4   events_played         219 non-null    int64  
 5   first_place           219 non-null    int64  
 6   second_place          219 non-null    int64  
 7   top_10                219 non-null    int64  
 8   top_25                219 non-null    int64  
 9   cuts_made             219 non-null    int64  
 10  fed_ex_rank           219 non-null    int64  
 11  strokes_gained_total  219 non-null    float64
dtypes: float64(1), int64(8), str(3)
memory usage: 20.7 KB
<class 'pandas.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 10 columns):
 #   Column           Non-Null Count  D

In [76]:
df_main["country"] = df_main["country"].str.title()
df_main.head()

,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total
0,Rory,McIlroy,Northern Ireland,2,14,1,1,6,9,14,4,1.341
1,Jackson,Suber,United States,94,20,0,0,3,6,13,74,0.475
2,Nicolai,Højgaard,Denmark,42,21,0,2,4,8,16,35,0.691
3,Kris,Ventura,Norway,195,19,0,1,3,5,7,107,0.195
4,Gordon,Sargent,United States,331,18,0,1,2,3,7,132,-0.694


In [77]:
events = df_main["events_played"].where(df_main["events_played"] > 0)

df_main["win_pct"] = (df_main["first_place"] / events * 100).round(2)
df_main["top_10_pct"] = (df_main["top_10"] / events * 100).round(2)

df_main.head()


,first_name,last_name,country,world_rank,events_played,first_place,second_place,top_10,top_25,cuts_made,fed_ex_rank,strokes_gained_total,win_pct,top_10_pct
0,Rory,McIlroy,Northern Ireland,2,14,1,1,6,9,14,4,1.341,7.14,42.86
1,Jackson,Suber,United States,94,20,0,0,3,6,13,74,0.475,0.00,15.00
2,Nicolai,Højgaard,Denmark,42,21,0,2,4,8,16,35,0.691,0.00,19.05
3,Kris,Ventura,Norway,195,19,0,1,3,5,7,107,0.195,0.00,15.79
4,Gordon,Sargent,United States,331,18,0,1,2,3,7,132,-0.694,0.00,11.11


# ------------ LOAD ----------------

In [79]:
MY_SQL_HOST = os.getenv("MYSQL_HOST")
MY_SQL_PORT = os.getenv("MYSQL_PORT")
MY_SQL_USER = os.getenv("MYSQL_USER")
MY_SQL_PASSWORD = os.getenv("MYSQL_PASSWORD")
MY_SQL_DATABASE = os.getenv("MYSQL_DATABASE")

engine = create_engine(
    f"mysql+pymysql://{MY_SQL_USER}:{MY_SQL_PASSWORD}@{MY_SQL_HOST}:{MY_SQL_PORT}/{MY_SQL_DATABASE}"
)


table_name = 'players_main'
df_main.to_sql(table_name,engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table {table_name} in database {MY_SQL_DATABASE}")


table_name = 'players_statistics'
df_stats.to_sql(table_name,engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table {table_name} in database {MY_SQL_DATABASE}")

Data successfully loaded into table players_main in database pga_db
Data successfully loaded into table players_statistics in database pga_db
